# 01 - Business Understanding
## Fraud Detection Decision Engine
### IEEE-CIS Fraud Detection Dataset

---

## 1. Business Context

E-commerce and digital payment volumes are growing rapidly year over year. With this growth comes an increasing surface area for **online payment fraud**, particularly **card-not-present (CNP)** transactions where the physical card is not required.

Fraud detection is no longer optional -- it is a core business function. Organizations that fail to detect fraud effectively face compounding consequences across financial, operational, and reputational dimensions.

This project aims to build a **data-driven Fraud Decision Engine** using the IEEE-CIS dataset, which contains real-world e-commerce transaction data from Vesta Corporation.

## 2. Business Problem

When fraud goes undetected or is poorly managed, the impact cascades across three critical dimensions:

### 2.1 Financial Loss
- **Direct loss** from fraudulent transactions that are approved and settled
- **Chargeback costs** -- fees imposed by card networks when cardholders dispute fraudulent charges
- **Operational cost** of investigating and resolving fraud cases
- **Penalty fees** from payment processors when fraud rates exceed acceptable thresholds

### 2.2 Customer Trust
- Customers who experience fraud on a platform **lose confidence** and are less likely to return
- Overly aggressive fraud blocking (false positives) creates **friction for legitimate customers**, leading to cart abandonment and churn
- Customers expect their transactions to be both **seamless and secure** -- failing either erodes trust

### 2.3 Reputation
- High fraud rates damage **brand credibility** in the market
- Negative media coverage and social media complaints amplify reputational harm
- Partners, merchants, and payment networks may **downgrade or terminate relationships** with platforms that have poor fraud controls
- Regulatory scrutiny increases for organizations with repeated fraud incidents

## 3. Stakeholders

### Primary

| Stakeholder | Role | Interest |
|-------------|------|----------|
| **Fraud Risk Team** | Owns fraud detection rules and models | Needs accurate, interpretable fraud scores to take action on suspicious transactions |
| **Risk Management** | Oversees enterprise risk posture | Requires visibility into fraud trends, exposure levels, and model performance to manage overall risk |

### Secondary

| Stakeholder | Role | Interest |
|-------------|------|----------|
| **Customer Service** | Handles disputes, chargebacks, and customer complaints | Needs clear fraud disposition data to resolve cases quickly and reduce handling time |
| **Finance** | Manages financial impact and reporting | Needs fraud loss metrics, chargeback volumes, and cost projections for budgeting and compliance |
| **Customer** | End user making transactions | Expects a frictionless payment experience without being wrongly blocked, while trusting the platform to protect them from fraud |

## 4. Decision Framework

The fraud decision engine will classify each transaction into one of three outcomes based on the predicted risk score. This framework will be refined as the project progresses.

### Initial Decision Flow

```
                    ┌─────────────────────┐
                    │   Transaction In    │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   Risk Scoring      │
                    │   (Model Predict)   │
                    └──────────┬──────────┘
                               │
              ┌────────────────┼────────────────┐
              │                │                │
              ▼                ▼                ▼
     ┌────────────┐   ┌──────────────┐   ┌────────────┐
     │  APPROVE   │   │     OTP      │   │   MANUAL   │
     │            │   │ Verification │   │   REVIEW   │
     │ Low Risk   │   │ Medium Risk  │   │ High Risk  │
     └────────────┘   └──────────────┘   └────────────┘
```

### Decision Tiers

| Tier | Risk Level | Action | Description |
|------|-----------|--------|-------------|
| **Approve** | Low | Auto-approve | Transaction proceeds without additional friction |
| **OTP** | Medium | Request verification | Trigger one-time password or step-up authentication |
| **Manual Review** | High | Queue for analyst | Transaction held and routed to fraud analyst for manual decision |

> **Note:** Threshold boundaries between tiers will be determined during the modeling phase (Notebook 05) based on cost-benefit analysis and business risk appetite.

## 5. Decision Philosophy

### Core Principle

> Fraud detection is fundamentally a **risk management** problem, not a classification problem.

The model does not make the final decision. The model produces a **risk score**. The decision is made by **business rules** that interpret that score in context.

### Philosophy

- Not every suspicious transaction should be declined.
- Customer experience must be balanced with fraud prevention.
- Multiple weak behavioral signals are generally more reliable than a single strong indicator.
- The objective is to **minimize expected loss**, not to eliminate all fraud.

### How Decisions Are Made

```
         Transaction
              │
              ▼
     Behavior Analysis
       (Feature Signals)
              │
              ▼
        Risk Score
       (ML Model Output)
              │
              ▼
       Business Rule
     (Threshold + Context)
              │
       ┌─────┼─────────────┐
       │           │             │
       ▼           ▼             ▼
    Approve       OTP      Manual Review
```

**Key distinction:** The ML model is a tool that informs the decision. It is not the decision-maker. Business rules, thresholds, and human judgment remain in control.

## 6. Success Metrics

### Primary Metrics

These are the metrics that matter to the business. They directly translate model performance into operational and financial impact.

| # | Metric | Definition | Why It's Primary |
|---|--------|-----------|------------------|
| 1 | **Expected Financial Loss** | Estimated monetary loss from undetected fraud (false negatives x fraud amount) | The ultimate bottom-line metric -- this is what the business actually loses |
| 2 | **Fraud Detection Recall** | Percentage of actual fraud transactions correctly identified | Directly determines how much fraud slips through undetected |
| 3 | **False Positive Rate** | Percentage of legitimate transactions incorrectly flagged as fraud | Measures customer friction -- too high and we lose legitimate revenue and trust |

### Secondary Metrics

These support model development, tuning, and comparison. They are important for the data science process but are not business objectives in themselves.

| Metric | Purpose |
|--------|---------|
| **Precision** | Of all flagged transactions, how many are truly fraud? Guides manual review workload efficiency |
| **ROC-AUC** | Overall model discrimination ability across all thresholds -- useful for model comparison, not a business target |
| **Manual Review Rate** | Percentage of transactions routed to manual review -- directly impacts operational cost and scalability |

> **Note:** This is not a Kaggle competition. ROC-AUC is a model development tool, not a business goal. The primary metrics above reflect what actually matters to stakeholders.

## 7. Dataset Description

The project uses the **IEEE-CIS Fraud Detection** dataset, sourced from a Kaggle competition. The data represents real-world e-commerce transactions provided by Vesta Corporation.

### Data Files

| Dataset | Rows | Columns | Purpose |
|---------|------|---------|--------|
| `train_transaction.csv` | ~590K | 394 | Core transaction attributes and fraud labels |
| `train_identity.csv` | ~144K | 41 | Device and identity information linked by `TransactionID` |
| `test_transaction.csv` | ~506K | 393 | Transaction features for prediction (no `isFraud` label) |
| `test_identity.csv` | ~141K | 41 | Identity features for test set transactions |

### Target Variable
- **`isFraud`** -- Binary (0 = legitimate, 1 = fraud)
- Available only in `train_transaction.csv`

### Feature Groups

| Group | Columns | Description |
|-------|---------|-------------|
| **Transaction Core** | `TransactionID`, `TransactionDT`, `TransactionAmt` | Unique ID, time delta from reference point, transaction amount |
| **Product** | `ProductCD` | Product code category (W, H, C, S, R) |
| **Card** | `card1` -- `card6` | Card information (type, category, issuing bank, etc.) |
| **Address** | `addr1`, `addr2` | Billing region and country |
| **Distance** | `dist1`, `dist2` | Distance-based features |
| **Email** | `P_emaildomain`, `R_emaildomain` | Purchaser and recipient email domains |
| **Count (C)** | `C1` -- `C14` | Counting features (e.g., how many transactions share the same address) |
| **Timedelta (D)** | `D1` -- `D15` | Time-based delta features |
| **Match (M)** | `M1` -- `M9` | Match features (e.g., does the name on the card match the address?) |
| **Vesta (V)** | `V1` -- `V339` | Vesta-engineered ranking, counting, and other rich features |
| **Identity** | `id_01` -- `id_38` | Network and digital identity signals (IP, proxy, etc.) |
| **Device** | `DeviceType`, `DeviceInfo` | Device type (mobile/desktop) and detailed device info |

### Key Characteristics
- **Identity data is partial** -- only ~25% of transactions have matching identity records
- **Many features are anonymized** -- the V-features and id-features have no public description
- **Feature naming is obfuscated** for privacy and security reasons

## 8. Analytical Roadmap

```
     Business Problem           ←  01. Business Understanding (this notebook)
           │
           ▼
         EDA                    ←  02. Exploratory Data Analysis
           │
           ▼
   Hypothesis Generation        ←  02. Exploratory Data Analysis
           │
           ▼
   Feature Engineering          ←  03. Feature Engineering
           │
           ▼
     Fraud Modeling             ←  04. Modeling
           │
           ▼
    Decision Engine             ←  05. Decision Engine
           │
           ▼
  Business Recommendation       ←  06. Final Report
```

---

**Next: [02_Exploratory_Data_Analysis](./02_Exploratory_Data_Analysis.ipynb)** -- Deep-dive into the data to uncover patterns, validate assumptions, and generate hypotheses that will drive feature engineering.